# Fine-tuning Qwen 3 4B with Unsloth AI
This notebook fine-tunes Qwen 3 4B for text style transfer using your CoT dataset

## 1. Install Required Packages

In [ ]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install or uv pip install
    !pip install unsloth vllm
else:
    pass # For Colab / Kaggle, we need extra instructions hidden below \/

In [ ]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !pip install unsloth vllm
else:
    try: import numpy; get_numpy = f"numpy=={numpy.__version__}"
    except: get_numpy = "numpy"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    get_vllm, get_triton = ("vllm==0.9.2", "triton==3.2.0") if is_t4 else ("vllm==0.10.2", "triton")
    !uv pip install -qqq --upgrade \
        unsloth {get_vllm} {get_numpy} torchvision bitsandbytes xformers
    !uv pip install -qqq {get_triton}
!uv pip install transformers==4.55.4
!uv pip install --no-deps trl==0.22.2

In [ ]:
!pip install mlflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.1/329.1 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.2/86.2 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 718.4/718.4 kB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.4/203.4 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.6/106.6 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

## 2. Import Libraries

In [ ]:
from unsloth import FastLanguageModel
import torch
import json
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from pathlib import Path
import mlflow
import mlflow.pytorch
import os
import pandas as pd

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 10-05 13:38:44 [__init__.py:244] Automatically detected platform cuda.
ERROR 10-05 13:38:46 [fa_utils.py:57] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
🦥 Unsloth Zoo will now patch everything to make training faster!


## 3. Configuration

In [ ]:
# Model configuration
max_seq_length = 2048
dtype = None  # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True  # Use 4bit quantization to reduce memory usage

# LoRA configuration
lora_r = 16
lora_alpha = 16
lora_dropout = 0
target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Training configuration
per_device_train_batch_size = 4
gradient_accumulation_steps = 4
num_train_epochs = 3
learning_rate = 2e-4
warmup_steps = 20
logging_steps = 10
save_steps = 100
eval_steps = 100


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
mlflow_dir = "/content/drive/MyDrive/mlruns"  # Change path as needed
os.makedirs(mlflow_dir, exist_ok=True)

# Set tracking URI
mlflow.set_tracking_uri(f"file://{mlflow_dir}")

# Set experiment
experiment_name = "qwen3-style-transfer-finetuning"
mlflow.set_experiment(experiment_name)

# Disable async logging
os.environ["MLFLOW_ENABLE_ASYNC_LOGGING"] = "false"

print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Saving to: {mlflow_dir}")

2025/10/05 13:39:59 INFO mlflow.tracking.fluent: Experiment with name 'qwen3-style-transfer-finetuning' does not exist. Creating a new experiment.


MLflow tracking URI: file:///content/drive/MyDrive/mlruns
Saving to: /content/drive/MyDrive/mlruns


## 4. Load Model with Unsloth

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# Note: If you specifically need Qwen3-4B, you can use:
# Unsloth optimizes the model automatically

==((====))==  Unsloth 2025.10.1: Fast Qwen3 patching. Transformers: 4.55.4. vLLM: 0.9.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


## 5. Add LoRA Adapters

In [ ]:
2model = FastLanguageModel.get_peft_model(
    model,
    r=lora_r,
    target_modules=target_modules,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Unsloth's optimized gradient checkpointing
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

Unsloth 2025.10.1 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


## 6. Load and Prepare Dataset

In [ ]:
def load_cot_dataset(file_path):
    """
    Load the CoT dataset from JSONL file
    """
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                entry = json.loads(line)
                data.append(entry)
            except json.JSONDecodeError:
                continue
    return data

# Load dataset
dataset_path = "/content/cot_dataset.jsonl"
raw_data = load_cot_dataset(dataset_path)
print(f"Loaded {len(raw_data)} examples")

# Show sample
if raw_data:
    print("\nSample entry:")
    print(json.dumps(raw_data[0], indent=2))

Loaded 3000 examples

Sample entry:
{
  "timestamp": "2025-09-09T22:35:14.758172",
  "input": "the term \"american\" is frequently misused to mean a citizen of the united states of america, despite the fact that anyone who lives in the americas is technically an \"american\".",
  "style": "Neutral, unbiased",
  "prompt": "\nYou are an expert at text style transfer. Your task is to transform text from one style to another. First, explain your reasoning about what needs to be changed, then provide the transformed text.\nOutput Format\n\n    Provide reasoning about the style elements that need to be changed\n    Mark your final transformed text with [Transferred]:\n\nExample 1: Informal \u2192 Formal\nSource Text: \"hey can u help me out with this thing?\" \nTarget Style: formal\n\nThe original text is informal. The use of \"hey\" is a casual greeting, \"u\" is text speak for \"you\", \"help me out\" is casual phrasing, and \"this thing\" is vague. To make it formal, I need to use proper 

## 7. Format Dataset for Training

In [ ]:
def format_prompt(entry):
    """
    Format the dataset entry into a chat format for Qwen
    """
    # Extract the prompt and response
    user_message = entry['prompt']
    assistant_response = entry['response']

    # Format as chat messages
    messages = [
        {"role": "user", "content": user_message},
        {"role": "assistant", "content": assistant_response}
    ]

    # Apply chat template
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": text}

# Format all entries
formatted_data = [format_prompt(entry) for entry in raw_data]

# Create HuggingFace dataset
dataset = Dataset.from_list(formatted_data)

# Split into train and eval
split_dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset['train']
eval_dataset = split_dataset['test']

print(f"Training examples: {len(train_dataset)}")
print(f"Evaluation examples: {len(eval_dataset)}")
print("\nSample formatted text:")
print(train_dataset[0]['text'][:500] + "...")

Training examples: 2700
Evaluation examples: 300

Sample formatted text:
<|im_start|>user

You are an expert at text style transfer. Your task is to transform text from one style to another. First, explain your reasoning about what needs to be changed, then provide the transformed text.
Output Format

    Provide reasoning about the style elements that need to be changed
    Mark your final transformed text with [Transferred]:

Example 1: Informal → Formal
Source Text: "hey can u help me out with this thing?" 
Target Style: formal

The original text is informal. The ...


## 8.Set up ML flow traking

In [ ]:
# mlflow.set_tracking_uri(f"file://{mlflow_artifacts_dir}")
# mlflow.set_experiment(experiment_name)

# # Start MLflow run
# run_name = f"qwen3_finetuning_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}"
# mlflow.start_run(run_name=run_name)

# print(f"MLflow run started: {run_name}")
# print(f"Artifacts will be saved to: {mlflow_artifacts_dir}")

# # Log parameters
# mlflow.log_params({
#     "max_seq_length": max_seq_length,
#     "load_in_4bit": load_in_4bit,
#     "lora_r": lora_r,
#     "lora_alpha": lora_alpha,
#     "lora_dropout": lora_dropout,
#     "per_device_train_batch_size": per_device_train_batch_size,
#     "gradient_accumulation_steps": gradient_accumulation_steps,
#     "num_train_epochs": num_train_epochs,
#     "learning_rate": learning_rate,
#     "warmup_steps": warmup_steps,
# })

## 9. Setup Trainer

In [ ]:
from transformers import TrainerCallback

class PrinterCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            if 'loss' in logs:
                print(f"Step {state.global_step}: Train Loss = {logs['loss']:.4f}")
            if 'eval_loss' in logs:
                print(f"Step {state.global_step}: Eval Loss = {logs['eval_loss']:.4f}")

In [ ]:
training_args = TrainingArguments(
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    warmup_steps=warmup_steps,
    num_train_epochs=num_train_epochs,
    learning_rate=learning_rate,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=logging_steps,
    logging_first_step=True,  # Add this to see first step
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    output_dir="outputs",
    save_steps=save_steps,
    eval_steps=eval_steps,
    eval_strategy="steps",  # Keep as-is
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="mlflow",
    run_name=f"qwen3_style-transfer_finetune_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
    callbacks=[PrinterCallback()],
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/2700 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/300 [00:00<?, ? examples/s]

## 9. Train the Model

In [ ]:
4# Start training
trainer_stats = trainer.train()

print("\nTraining completed!")
print(f"Training loss: {trainer_stats.training_loss}")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,700 | Num Epochs = 3 | Total steps = 507
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)
2025/10/05 13:47:50 ERROR mlflow.utils.async_logging.async_logging_queue: Run Id 54194f16a2ce4a15b3fb3dd375937b51: Failed to log run data: Exception: Changing param values is not allowed. Param with key='logging_dir' was already logged with value='outputs/runs/Oct05_13-42-14_b19e53d8d09e' for run ID='54194f16a2ce4a15b3fb3dd375937b51'. Attempted logging new value 'outputs/runs/Oct05_13-47-07_b19e53d8d09e'.


Step,Training Loss,Validation Loss
100,0.436200,0.451405
200,0.388500,0.432593
300,0.391300,0.425025
400,0.365000,0.423920
500,0.347200,0.421513


Step 1: Train Loss = 1.5885
Step 10: Train Loss = 1.4914
Step 20: Train Loss = 0.8857
Step 30: Train Loss = 0.5498
Step 40: Train Loss = 0.5187
Step 50: Train Loss = 0.4909
Step 60: Train Loss = 0.4748
Step 70: Train Loss = 0.4553
Step 80: Train Loss = 0.4567
Step 90: Train Loss = 0.4425


Unsloth: Not an error, but Qwen3ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Step 100: Train Loss = 0.4362
Step 100: Eval Loss = 0.4514
Step 110: Train Loss = 0.4302
Step 120: Train Loss = 0.4397
Step 130: Train Loss = 0.4331
Step 140: Train Loss = 0.4248
Step 150: Train Loss = 0.4283
Step 160: Train Loss = 0.4193
Step 170: Train Loss = 0.4114
Step 180: Train Loss = 0.4103
Step 190: Train Loss = 0.4147
Step 200: Train Loss = 0.3885
Step 200: Eval Loss = 0.4326
Step 210: Train Loss = 0.4119
Step 220: Train Loss = 0.3953
Step 230: Train Loss = 0.4017
Step 240: Train Loss = 0.4058
Step 250: Train Loss = 0.4022
Step 260: Train Loss = 0.3984
Step 270: Train Loss = 0.3889
Step 280: Train Loss = 0.4013
Step 290: Train Loss = 0.4043
Step 300: Train Loss = 0.3913
Step 300: Eval Loss = 0.4250
Step 310: Train Loss = 0.4122
Step 320: Train Loss = 0.3920
Step 330: Train Loss = 0.3759
Step 340: Train Loss = 0.3747
Step 350: Train Loss = 0.3777
Step 360: Train Loss = 0.3726
Step 370: Train Loss = 0.3779
Step 380: Train Loss = 0.3715
Step 390: Train Loss = 0.3698
Step 400: Tra

## 10. Save the Model

In [ ]:
# Save LoRA adapters
model.save_pretrained("qwen3_style_transfer_lora")
tokenizer.save_pretrained("qwen3_style_transfer_lora")

print("Model saved to 'qwen3_style_transfer_lora'")

# Optional: Save merged model (full model + LoRA)
# model.save_pretrained_merged("qwen3_style_transfer_merged", tokenizer, save_method="merged_16bit")
# print("Merged model saved to 'qwen3_style_transfer_merged'")

## 11. Test the Fine-tuned Model

In [ ]:
# Enable inference mode
FastLanguageModel.for_inference(model)

def test_style_transfer(text, target_style="non-toxic"):
    """
    Test the fine-tuned model on a sample text
    """
    prompt = f"""You are an expert at text style transfer. Your task is to transform text from one style to another. First, explain your reasoning about what needs to be changed, then provide the transformed text.

Source Text: "{text}"
Target Style: {target_style}
"""

    messages = [
        {"role": "user", "content": prompt}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result

# Test examples
test_texts = [
    "you're such an idiot for thinking that",
    "this is fucking terrible",
    "get lost, nobody wants you here"
]

print("Testing fine-tuned model:\n")
for text in test_texts:
    print(f"Original: {text}")
    result = test_style_transfer(text)
    print(f"Transformed:\n{result}\n")
    print("-" * 80)

## 12. Load Saved Model for Future Use

In [ ]:
# To load the model later:
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name="qwen3_style_transfer_lora",
#     max_seq_length=max_seq_length,
#     dtype=dtype,
#     load_in_4bit=load_in_4bit,
# )
# FastLanguageModel.for_inference(model)